# Agent Control (Phase 6)

This notebook implements the **Control agent**.

Phase 6 scope:
- Subscribe to Trigger person-state and Observer exposure-event topics
- Apply one-way transition rules (`susceptible -> infected -> recovered`)
- Keep `recovered` terminal
- Publish health updates to `${base_topic}/pandemic/control/health_update`

In [1]:
# Cell purpose: Import modules, load config, and define Phase 6 control topic contracts.
from __future__ import annotations

import json
from random import Random

import simulated_city.mqtt as mqtt
from simulated_city.config import load_config
from simulated_city.epidemic import (
    build_health_update,
    can_transition,
    days_infected,
    parse_exposure_event,
    parse_person_state,
    recovery_steps,
)

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

if sim_cfg is None:
    raise ValueError("Phase 6 requires a 'simulation' section in config.yaml")

base_topic = mqtt_cfg.base_topic
topic_person_state = f"{base_topic}/pandemic/trigger/person_state"
topic_exposure_event = f"{base_topic}/pandemic/observer/exposure_event"
topic_health_update = f"{base_topic}/pandemic/control/health_update"

infection_probability = sim_cfg.infection_probability
simulated_hours_per_step = sim_cfg.simulated_hours_per_step
recovery_after_steps = recovery_steps(sim_cfg.recovery_days, simulated_hours_per_step)
seed = sim_cfg.seed if sim_cfg.seed is not None else 42
rng = Random(seed + 600)

print(
    f"Control config loaded. infection_probability={infection_probability}, "
    f"recovery_after_steps={recovery_after_steps}"
)
print(
    f"Control topics: subscribe=[{topic_person_state}, {topic_exposure_event}], "
    f"publish={topic_health_update}"
)

Control config loaded. infection_probability=1.0, recovery_after_steps=960
Control topics: subscribe=[simulated-city/pandemic/trigger/person_state, simulated-city/pandemic/observer/exposure_event], publish=simulated-city/pandemic/control/health_update


In [2]:
# Cell purpose: Define control state containers and transition functions.
person_status: dict[str, str] = {}
infected_since_step: dict[str, int] = {}
last_step_seen = -1
health_updates_published = 0
malformed_messages = 0
client = None


def publish_health_update(*, step: int, ts: str, person_id: str, from_status: str, to_status: str, reason: str) -> None:
    global health_updates_published
    if not can_transition(from_status, to_status):
        return

    if from_status == "infected":
        start_step = infected_since_step.get(person_id, step)
        infected_days = days_infected(step, start_step, simulated_hours_per_step)
    else:
        infected_days = 0.0

    payload = build_health_update(
        step=step,
        ts=ts,
        person_id=person_id,
        from_status=from_status,
        to_status=to_status,
        reason=reason,
        days_infected=infected_days,
    )
    mqtt.publish_json_checked(client, topic_health_update, payload)
    health_updates_published += 1


def maybe_recover_person(person_id: str, current_step: int, ts: str) -> None:
    status = person_status.get(person_id)
    if status != "infected":
        return

    infected_step = infected_since_step.get(person_id, current_step)
    if current_step - infected_step < recovery_after_steps:
        return

    person_status[person_id] = "recovered"
    publish_health_update(
        step=current_step,
        ts=ts,
        person_id=person_id,
        from_status="infected",
        to_status="recovered",
        reason="recovery",
    )

In [3]:
# Cell purpose: Connect control MQTT client, process Trigger+Observer messages, and publish health updates.
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="control")
print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")


def on_message(client_obj, userdata, msg):
    global last_step_seen, malformed_messages

    try:
        payload = json.loads(msg.payload.decode())
    except Exception:
        malformed_messages += 1
        return

    if msg.topic == topic_person_state:
        try:
            person_msg = parse_person_state(payload)
        except Exception:
            malformed_messages += 1
            return

        person_id = person_msg.person_id
        step = person_msg.step
        ts = person_msg.ts
        incoming_status = person_msg.health_status

        if person_id not in person_status:
            person_status[person_id] = incoming_status
            if incoming_status == "infected":
                infected_since_step[person_id] = step

        maybe_recover_person(person_id, step, ts)
        last_step_seen = max(last_step_seen, step)
        return

    if msg.topic == topic_exposure_event:
        try:
            exposure = parse_exposure_event(payload)
        except Exception:
            malformed_messages += 1
            return

        if not exposure.within_radius:
            return

        target_id = exposure.target_id
        current_status = person_status.get(target_id, "susceptible")
        if current_status != "susceptible":
            return

        infection_roll = rng.random()
        if infection_roll <= infection_probability:
            person_status[target_id] = "infected"
            infected_since_step[target_id] = exposure.step
            publish_health_update(
                step=exposure.step,
                ts=exposure.ts,
                person_id=target_id,
                from_status="susceptible",
                to_status="infected",
                reason="infection",
            )


client.on_message = on_message
client.subscribe(topic_person_state)
client.subscribe(topic_exposure_event)
print(f"Subscribed to {topic_person_state}")
print(f"Subscribed to {topic_exposure_event}")
print("Control is now listening. Run Trigger + Observer to feed this agent.")

Connected to MQTT broker at 127.0.0.1:1883
Subscribed to simulated-city/pandemic/trigger/person_state
Subscribed to simulated-city/pandemic/observer/exposure_event
Control is now listening. Run Trigger + Observer to feed this agent.


In [4]:
# Cell purpose: Print control summary counters and disconnect cleanly when finished.
infected_count = sum(1 for s in person_status.values() if s == "infected")
recovered_count = sum(1 for s in person_status.values() if s == "recovered")
susceptible_count = sum(1 for s in person_status.values() if s == "susceptible")

print(
    f"Control summary: tracked_people={len(person_status)}, susceptible={susceptible_count}, "
    f"infected={infected_count}, recovered={recovered_count}, "
    f"health_updates_published={health_updates_published}, malformed_messages={malformed_messages}, "
    f"last_step_seen={last_step_seen}"
)

connector = getattr(client, "_simulated_city_connector", None)
if connector is not None:
    connector.disconnect()
    print("Disconnected from MQTT broker.")
else:
    print("No connector reference found; client disconnect skipped.")

Control summary: tracked_people=200, susceptible=40, infected=56, recovered=104, health_updates_published=259, malformed_messages=0, last_step_seen=1436
Disconnected from MQTT broker.
